# 02.7 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/02-regression/07-decision-tree-regressor/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
six = pd.DataFrame({"customer": list("ABCDEF"),
                    "days_since_last": [1, 2, 3, 10, 20, 30],
                    "spend": [120, 100, 110, 40, 30, 20]})
six

from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
def var(a): return ((a - a.mean())**2).mean()

## Exercise 1

Parent {120, 100, 110}: mean 110, Var 66.7.

- Threshold 1.5: left {120} Var 0; right {100, 110} mean 105, Var 25. Weighted $\frac{1}{3}0 + \frac{2}{3}25 = 16.7$. Δ = 50.
- Threshold 2.5: left {120, 100} mean 110, Var 100; right {110} Var 0. Weighted $\frac{2}{3}100 + \frac{1}{3}0 = 66.7$. Δ = 0.

Threshold **1.5** wins (Δ = 50). Splitting at 2.5 achieves nothing because the two left rows are the most different pair.

In [2]:
L = np.array([120, 100, 110.]); xl = np.array([1, 2, 3.])
for t in [1.5, 2.5]:
    a, b = L[xl <= t], L[xl > t]
    print(t, "Δ =", round(var(L) - (len(a)*var(a) + len(b)*var(b)) / 3, 1))

1.5 Δ = 50.0
2.5 Δ = 0.0


## Exercise 2

In [3]:
cust20 = pd.read_csv(data_path("shopwave_customers_small.csv"))
feats = ["visits_last_month", "items_last_month", "days_since_last"]; y20 = cust20["spend_next_month"]
for d in range(1, 7):
    t = DecisionTreeRegressor(max_depth=d, random_state=0).fit(cust20[feats], y20)
    print(f"depth {d}: leaves {t.get_n_leaves():>2}  training MAE {mean_absolute_error(y20, t.predict(cust20[feats])):.2f}")

depth 1: leaves  2  training MAE 31.23
depth 2: leaves  4  training MAE 16.74
depth 3: leaves  8  training MAE 7.48
depth 4: leaves 13  training MAE 3.53
depth 5: leaves 17  training MAE 1.03
depth 6: leaves 20  training MAE 0.00


MAE reaches 0 once there are as many leaves as distinct rows (around depth 5–6 here). Zero training error means the tree has stored the table, not learned from it; on a new customer it would do no better than a shallow tree and probably worse.

## Exercise 3

In [4]:
X = cust20[["visits_last_month"]]
t2 = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X, y20)
ln = LinearRegression().fit(X, y20)
new = pd.DataFrame({"visits_last_month": [25]})
print("tree:", t2.predict(new).round(1), "  line:", ln.predict(new).round(1))
print("max visits seen in training:", X.max().item())

tree: [216.]   line: [448.9]
max visits seen in training: 14


The tree returns the mean of its top leaf (customers with the most visits it has seen); the line extends the slope to ~£450. Neither is *known* to be right (nobody in the data visited 25 times), but the line's answer at least moves in the right direction. The honest answer is "outside the data, do not trust either; flag it".

## Exercise 4

In [5]:
import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail
raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
def clean_retail(df):
    out = df[~df["Invoice"].astype(str).str.startswith("C")]
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates().copy()
    out["Customer ID"] = out["Customer ID"].astype(int); out["Revenue"] = out["Quantity"] * out["Price"]
    return out
sales = clean_retail(raw)
cutoff = pd.Timestamp("2011-09-01")
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]
cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
    products=("StockCode", "nunique"), first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["avg_order"] = cust["spend"] / cust["orders"]
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
cust["spend_next_quarter"] = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
feats_real = ["orders", "spend", "products", "avg_order", "tenure_days", "days_since_last"]
rule = DecisionTreeRegressor(max_depth=3, min_samples_leaf=50, random_state=0).fit(cust[feats_real], cust["spend_next_quarter"])
print(export_text(rule, feature_names=feats_real, show_weights=True))

|--- spend <= 10640.18
|   |--- spend <= 3293.64
|   |   |--- spend <= 1122.67
|   |   |   |--- value: [268.28]
|   |   |--- spend >  1122.67
|   |   |   |--- value: [832.15]
|   |--- spend >  3293.64
|   |   |--- spend <= 5926.43
|   |   |   |--- value: [1723.63]
|   |   |--- spend >  5926.43
|   |   |   |--- value: [3503.97]
|--- spend >  10640.18
|   |--- value: [17576.64]



Rules will read roughly like:

1. **Past spend above ~£10,600**: expect well over £10,000 next quarter; an account manager calls every one.
2. **Past spend £3,300–10,600**: expect £1,700–3,500; worth a call if the list is short.
3. **Past spend below £3,300**: expect a few hundred at most, less if under ~£1,100; email, not phone.

Every split is on `spend`: with 50 customers per leaf, no other feature reduced variance enough to be chosen. The tree is telling you that, for next-quarter pounds, past pounds is almost the whole story.